# Level 1 — is the interpolated 9-point predictive CDF calibrated at every 5% rung?

The product card prints **"~C% chance this day will be `<predicate>`"**. C is an
integral of the forecast's **9-point CQR-calibrated band** treated as a
piecewise-linear predictive CDF. As of the q9 retrain **all 9 points are trained
quantile heads** (levels .01/.05/.10/.25/.50/.75/.90/.95/.99) — the interior
q25/q75 used to be probit-interpolated client-side and now come off the table.
The card's `invCdf` still reads values at arbitrary 5% rungs that are themselves
linear interpolations *between* those 9 points.

**Question (Level 1 of the validation relay):** build the forecast band exactly
as the frontend does, and ask — is that piecewise-linear CDF calibrated at every
5% rung, especially the **12 interpolated rungs**
(15,20,30,35,40,45,55,60,65,70,80,85)? Rungs 25 and 75 moved from interpolated
to trained in the q9 retrain, so they are now Level-0 anchors, not news.

**Honest split.** The held-out **test** month-blocks are cut into two disjoint
halves — *calib* blocks fit the per-level CQR shifts (exactly like production),
*eval* blocks measure coverage. Neither the CatBoost fit nor the CQR shifts ever
saw the eval blocks, so rung coverage there is an out-of-sample read. Level 0
(`cqr_per_level_validity.ipynb`) already showed the 9 trained heads cover at
their nominal levels; the 12 interpolated rungs are the news here.

In [1]:
import numpy as np, pandas as pd
from pathlib import Path
from catboost import CatBoostRegressor
import train_quantile_debias as tqd
import make_debias_tables as mdt   # reuse the PRODUCTION per-level CQR shift + isotonic

pd.set_option("display.width", 170, "display.float_format", lambda v: f"{v:.3f}")

TAG = "qn8620_s0_q9"
QUANTILE_LEVELS = tqd.QUANTILES                  # [.01 .05 .10 .25 .50 .75 .90 .95 .99]
I50 = QUANTILE_LEVELS.index(0.50)
# services/tieredData.ts / confidence.ts PRECIP_TRACE_MM — a value <1mm reads as 0mm.
PRECIP_TRACE_MM = 1.0

# The 9 CDF points the frontend integrates — now ALL trained heads, so the band
# columns map 1:1 onto QUANTILE_LEVELS in order (asserted, since the whole
# notebook indexes one by the other).
BAND_P = np.array([0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99])
BAND_COLS = ["q01", "lo", "q10", "q25", "mid", "q75", "q90", "hi", "q99"]
assert np.allclose(BAND_P, QUANTILE_LEVELS), "band points must equal the trained levels"

# rung ladder the card reads off the CDF; which of those are TRAINED heads.
RUNGS = list(range(5, 100, 5))                               # 5,10,...,95
TRAINED_RUNGS = {5, 10, 25, 50, 75, 90, 95}                  # multiples of 5 that are trained
INTERP_RUNGS = [r for r in RUNGS if r not in TRAINED_RUNGS]  # the 12 interpolated rungs

# cache is keyed by TAG: the 7-level bands from the previous run live alongside
# these and MUST NOT be picked up by a skip-if-exists load.
CACHE_DIR = Path("data/nb_confidence"); CACHE_DIR.mkdir(parents=True, exist_ok=True)
print("tag:", TAG)
print("quantile levels:", QUANTILE_LEVELS)
print("9 CDF points p =", BAND_P.tolist())
print(f"trained rungs: {sorted(TRAINED_RUNGS)}")
print(f"interpolated rungs ({len(INTERP_RUNGS)}):", INTERP_RUNGS)

tag: qn8620_s0_q9
quantile levels: [0.01, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99]
9 CDF points p = [0.01, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99]
trained rungs: [5, 10, 25, 50, 75, 90, 95]
interpolated rungs (12): [15, 20, 30, 35, 40, 45, 55, 60, 65, 70, 80, 85]


In [2]:
# --- load the cached post-split processed frame (features + split already baked) ---
frame = pd.read_feather("data/pipeline_frame_full.feather")
print(f"{len(frame):,} rows, {frame.key.nunique()} cells, "
      f"{frame.date.min().date()}..{frame.date.max().date()}")

# GUARD: a cached frame built before the fc_version mask-order fix carries WRONG
# cycle labels (the 49r1 mask overwrote every 50r1 row), and a model trained with
# the fix would then be evaluated on data the fix never touched. Recompute the
# labels from the frame's own dates and refuse a frame that disagrees.
def assert_fc_version_current(frame):
    expected = pd.Series(tqd.FC_BASE, index=frame.index)
    for change_date, cycle in sorted(tqd.FC_CHANGES):
        expected = expected.mask(frame["date"] >= change_date, cycle)
    actual = frame["fc_version"].astype(str)
    bad = int((actual != expected.astype(str)).sum())
    assert bad == 0, (
        f"cached frame has STALE fc_version labels ({bad:,} rows disagree; "
        f"cached={dict(actual.value_counts())}). Delete "
        f"data/pipeline_frame_full.feather and rebuild from source."
    )
    print("fc_version guard OK:", dict(actual.value_counts()))

assert_fc_version_current(frame)

# monthly block id exactly as tqd.split builds it, so the TEST blocks split into
# disjoint calib vs eval halves (copied from q25_q75_interp_check.ipynb cell 2).
origin = frame.date.min().normalize()
frame["block"] = ((frame.date.dt.year - origin.year) * 12
                  + (frame.date.dt.month - origin.month))
test_blocks = np.sort(frame.loc[frame.role == "test", "block"].unique())
calib_blocks = set(test_blocks[0::2])                        # even-indexed -> calib
eval_blocks  = set(test_blocks[1::2])                        # odd-indexed  -> eval
print(f"test blocks: {list(test_blocks)}")
print(f"  calib -> {sorted(calib_blocks)}")
print(f"  eval  -> {sorted(eval_blocks)}")

7,609,855 rows, 8620 cells, 2024-03-01..2026-07-31


fc_version guard OK: {'49r1': np.int64(4706520), '48r1': np.int64(2206720), '50r1': np.int64(696615)}


test blocks: [np.int32(0), np.int32(5), np.int32(10), np.int32(15), np.int32(20), np.int32(25)]
  calib -> [np.int32(0), np.int32(10), np.int32(20)]
  eval  -> [np.int32(5), np.int32(15), np.int32(25)]


In [3]:
# --- exact ports of the frontend CDF primitives (port correctness is load-bearing
#     for this notebook AND Notebook 2, so cell 5 asserts them) ------------------

def precip_trace_clamp(values):
    "confidence.ts bandQuantilePoints / tieredData: a value <1mm reads as 0mm."
    values = np.asarray(values, float)
    return np.where(values < PRECIP_TRACE_MM, 0.0, values)

def value_at_tail_fraction(values, q, is_high_side):
    "confidence.ts:122 order-statistic boundary; sorts a copy, clamps the index."
    sorted_v = np.sort(np.asarray(values, float))
    n = len(sorted_v)
    idx = int(np.floor((1 - q) * n)) if is_high_side else int(np.ceil(q * n)) - 1
    return float(sorted_v[min(max(idx, 0), n - 1)])

def prob_in_interval(point_v, point_p, lo, hi):
    "confidence.ts:70 — mass of the 9-point CDF in [lo,hi]; point masses + outer tails."
    v = np.asarray(point_v, float); p = np.asarray(point_p, float)
    n = len(v); total = 0.0
    if lo <= v[0]  <= hi: total += p[0]                      # mass below p01
    if lo <= v[-1] <= hi: total += 1 - p[-1]                 # mass above p99
    for i in range(n - 1):
        va, vb = v[i], v[i + 1]; dp = p[i + 1] - p[i]
        if vb == va:
            if lo <= va <= hi: total += dp                   # point mass at va
        else:
            lo_c = min(max(lo, va), vb); hi_c = min(max(hi, va), vb)
            total += dp * ((hi_c - lo_c) / (vb - va))
    return float(min(max(total, 0.0), 1.0))

def inv_cdf_col(band, p_points, u):
    "HistogramChart.invCdf, vectorized over rows for one probability u. band: (n,9)."
    if u <= p_points[0]:  return band[:, 0].copy()
    if u >= p_points[-1]: return band[:, -1].copy()
    i = int(np.searchsorted(p_points, u, side="right") - 1)  # p[i] <= u < p[i+1]
    pa, pb = p_points[i], p_points[i + 1]
    va, vb = band[:, i], band[:, i + 1]
    return va.copy() if pb == pa else va + ((u - pa) / (pb - pa)) * (vb - va)

In [4]:
# --- port sanity: hand-built cases, assert (do not eyeball) ---------------------
non_degenerate = np.array([0., 1, 2, 3, 4, 5, 6, 7, 8])     # a strictly-rising band
# 1. prob_in_interval: fully enclosed -> ~1.0 (outer tails counted, not 0.98)
assert abs(prob_in_interval(non_degenerate, BAND_P, -1, 9) - 1.0) < 1e-9
# an interior bracket [mid, q90] == [4, 6] spans p .50->.90 = .40 mass
assert abs(prob_in_interval(non_degenerate, BAND_P, 4, 6) - 0.40) < 1e-9
# one-sided high tail [q90, inf) -> 1 - .90 = .10
assert abs(prob_in_interval(non_degenerate, BAND_P, 6, np.inf) - 0.10) < 1e-9
# 2. point-mass spike: a bone-dry precip band (all zeros)
dry = np.zeros(9)
assert abs(prob_in_interval(dry, BAND_P, -1, 1) - 1.0) < 1e-9    # 0 inside  -> ~1
assert abs(prob_in_interval(dry, BAND_P, 0.5, 5) - 0.0) < 1e-9   # 0 excluded -> ~0
# 3. value_at_tail_fraction: behavioural check against the pool it slices
rng = np.random.default_rng(0); pool = rng.normal(size=5000)
for q in (0.05, 0.10, 0.20):
    hi_b = value_at_tail_fraction(pool, q, True)
    lo_b = value_at_tail_fraction(pool, q, False)
    assert abs(np.mean(pool >= hi_b) - q) < 1e-3    # ~q of the pool at/above the high cut
    assert abs(np.mean(pool <= lo_b) - q) < 1e-3    # ~q of the pool at/below the low cut
# 4. inv_cdf recovers the 9 anchors exactly and interpolates the interior linearly
band1 = non_degenerate.reshape(1, -1)
for j, u in enumerate(BAND_P):
    assert abs(inv_cdf_col(band1, BAND_P, u)[0] - non_degenerate[j]) < 1e-9
expect_15 = 2 + (0.15 - 0.10) / (0.25 - 0.10) * (3 - 2)     # between q10(=2) and q25(=3)
assert abs(inv_cdf_col(band1, BAND_P, 0.15)[0] - expect_15) < 1e-9
print("port sanity: all asserts passed")

port sanity: all asserts passed


In [5]:
# --- per-var calibrated 9-point band on EVAL rows, cached for reuse. Notebook 2
#     reads these feathers and never touches the models. Skip-if-exists. ---------

GATED = mdt.gated_keys(TAG, 0.1)          # {var -> set(cell keys whose ML band is gated out)}

def predict_sorted(model, X, chunk=200_000):
    "chunked 9-head predict (tmax model is ~400MB); sort heads per row as production does."
    out = [np.sort(np.asarray(model.predict(X.iloc[s:s + chunk])), axis=1)
           for s in range(0, len(X), chunk)]
    return np.concatenate(out, axis=0)

def build_eval_bands(name):
    "eval-row frame (key,date,hres,truth_abs,gated + 9 absolute band cols)."
    cache = CACHE_DIR / f"eval_bands_{name}_{TAG}.feather"
    if cache.exists():
        print(f"[{name}] load cache {cache.name}")
        return pd.read_feather(cache)

    var = next(v for v in tqd.VARS if v["name"] == name)
    nonneg = var["nonneg"]; hcol = tqd.hres_col(name)
    feat = tqd.feature_cols(name, with_cell=True, with_cross=False)
    rows = frame[(frame.role == "test") & frame[f"bias_{name}"].notna()]
    X = rows[feat].astype({"key": str, "fc_version": str})

    model = CatBoostRegressor(); model.load_model(str(tqd.MODELS / f"M3_base_{name}_{TAG}.cbm"))
    preds = predict_sorted(model, X)                          # (n,9) sorted bias-delta heads
    del model

    y    = rows[f"bias_{name}"].to_numpy()
    hres = rows[hcol].to_numpy()
    is_calib = rows.block.isin(calib_blocks).to_numpy()
    is_eval  = rows.block.isin(eval_blocks).to_numpy()

    # per-level CQR shift fit on CALIB only, median head stays 0 (mirror compute_level_shifts)
    shifts = np.zeros(len(QUANTILE_LEVELS))
    for i, level in enumerate(QUANTILE_LEVELS):
        if i != I50:
            shifts[i] = mdt.per_level_shift(y[is_calib] - preds[is_calib, i], level)

    # apply to EVAL exactly as production bakes the tables: round 2dp, re-isotonize
    adj = mdt.pin_median_isotonic(np.round(preds[is_eval] + shifts, 2))    # (m,9) bias deltas
    abs_heads = hres[is_eval][:, None] + adj                               # -> absolute units
    if nonneg:
        abs_heads = np.maximum(abs_heads, 0.0)                             # ci.ts nonneg floor

    # all nine points are trained heads now — no shoulder interpolation step
    band = np.round(abs_heads, 3)                                          # ci.ts round3

    truth_abs = hres[is_eval] + y[is_eval]
    if name == "precip":                                                  # trace clamp band + truth
        band = precip_trace_clamp(band)
        truth_abs = precip_trace_clamp(truth_abs)

    ev_key = rows.key.to_numpy()[is_eval]
    out = pd.DataFrame({
        "key":       ev_key,
        "date":      rows.date.to_numpy()[is_eval],
        "hres":      hres[is_eval],
        "truth_abs": truth_abs,
        "gated":     pd.Series(ev_key).astype(str).isin(GATED[name]).to_numpy(),
    })
    for j, c in enumerate(BAND_COLS):
        out[c] = band[:, j]

    # the 9 points must stay ascending after clamps (the CDF must be valid)
    assert (np.diff(out[BAND_COLS].to_numpy(), axis=1) >= -1e-9).all(), f"{name}: band not monotone"
    out.to_feather(cache)
    print(f"[{name}] built {len(out):,} eval rows -> {cache.name} (gated {out.gated.mean()*100:.1f}%)")
    return out

eval_bands = {v["name"]: build_eval_bands(v["name"]) for v in tqd.VARS}
{k: len(df) for k, df in eval_bands.items()}

[12:30:27]   gate tmax: 53 / 8620 cells dropped (damage > 0.1)


[12:30:27]   gate tmin: 18 / 8620 cells dropped (damage > 0.1)


[12:30:27]   gate precip: 351 / 8620 cells dropped (damage > 0.1)


[12:30:27]   gate wind: 5 / 8620 cells dropped (damage > 0.1)


[12:30:27]   gate dewpt: 48 / 8620 cells dropped (damage > 0.1)


[tmax] load cache eval_bands_tmax_qn8620_s0_q9.feather


[tmin] load cache eval_bands_tmin_qn8620_s0_q9.feather
[precip] load cache eval_bands_precip_qn8620_s0_q9.feather


[wind] load cache eval_bands_wind_qn8620_s0_q9.feather
[dewpt] load cache eval_bands_dewpt_qn8620_s0_q9.feather


{'tmax': 784420,
 'tmin': 784420,
 'precip': 784420,
 'wind': 784420,
 'dewpt': 784420}

In [6]:
# --- marginal rung coverage: empirical P(truth <= inv_cdf(band, r%)) vs r, in pp ---
def rung_coverage(df, rung_list=RUNGS):
    band = df[BAND_COLS].to_numpy(); truth = df.truth_abs.to_numpy()
    recs = []
    for r in rung_list:
        thr = inv_cdf_col(band, BAND_P, r / 100)
        cov = float(np.mean(truth <= thr))
        # flag rungs that land on a point-mass (flat) segment of the CDF
        i = min(max(int(np.searchsorted(BAND_P, r / 100, side="right") - 1), 0), len(BAND_P) - 2)
        flat_frac = float(np.mean(band[:, i + 1] - band[:, i] <= 1e-9))
        recs.append(dict(rung=r, kind=("trained" if r in TRAINED_RUNGS else "interp"),
                         cov_pct=round(cov * 100, 2), err_pp=round((cov - r / 100) * 100, 2),
                         flat_frac=round(flat_frac, 3)))
    return pd.DataFrame(recs)

rung_tables = {}
for name, df in eval_bands.items():
    tbl = rung_coverage(df); rung_tables[name] = tbl
    print(f"\n=== {name}: marginal rung coverage  (n={len(df):,}) ===")
    print(tbl.to_string(index=False))

# precip also on the WET-forecast subset (hres >= 1mm), where the point mass lifts
wet = eval_bands["precip"][eval_bands["precip"].hres >= 1.0]
rung_tables["precip_wet"] = rung_coverage(wet)
print(f"\n=== precip WET-forecast subset  (hres>=1mm, n={len(wet):,}) ===")
print(rung_tables["precip_wet"].to_string(index=False))


=== tmax: marginal rung coverage  (n=784,420) ===
 rung    kind  cov_pct  err_pp  flat_frac
    5 trained    4.480  -0.520      0.000
   10 trained    9.060  -0.940      0.000
   15  interp   12.750  -2.250      0.000
   20  interp   17.600  -2.400      0.000
   25 trained   23.760  -1.240      0.000
   30  interp   28.000  -2.000      0.000
   35  interp   32.730  -2.270      0.000
   40  interp   37.830  -2.170      0.000
   45  interp   43.260  -1.740      0.000
   50 trained   48.930  -1.070      0.000
   55  interp   54.440  -0.560      0.000
   60  interp   59.880  -0.120      0.000
   65  interp   65.130   0.130      0.000
   70  interp   70.110   0.110      0.000
   75 trained   74.620  -0.380      0.000
   80  interp   81.090   1.090      0.000
   85  interp   86.210   1.210      0.000
   90 trained   90.140   0.140      0.000
   95 trained   95.190   0.190      0.000

=== tmin: marginal rung coverage  (n=784,420) ===
 rung    kind  cov_pct  err_pp  flat_frac
    5 trained   


=== precip: marginal rung coverage  (n=784,420) ===
 rung    kind  cov_pct  err_pp  flat_frac
    5 trained   58.790  53.790      0.824
   10 trained   59.590  49.590      0.703
   15  interp   60.430  45.430      0.703
   20  interp   61.740  41.740      0.703
   25 trained   63.640  38.640      0.615
   30  interp   65.190  35.190      0.615
   35  interp   66.750  31.750      0.615
   40  interp   68.610  28.610      0.615
   45  interp   70.690  25.690      0.615
   50 trained   72.770  22.770      0.546
   55  interp   75.310  20.310      0.546
   60  interp   77.540  17.540      0.546
   65  interp   79.660  14.660      0.546
   70  interp   81.740  11.740      0.546
   75 trained   83.570   8.570      0.405
   80  interp   86.830   6.830      0.405
   85  interp   89.530   4.530      0.405
   90 trained   91.610   1.610      0.191
   95 trained   95.520   0.520      0.049

=== wind: marginal rung coverage  (n=784,420) ===
 rung    kind  cov_pct  err_pp  flat_frac
    5 trained 


=== dewpt: marginal rung coverage  (n=784,420) ===
 rung    kind  cov_pct  err_pp  flat_frac
    5 trained    4.970  -0.030      0.000
   10 trained   10.120   0.120      0.000
   15  interp   13.950  -1.050      0.000
   20  interp   18.940  -1.060      0.000
   25 trained   25.050   0.050      0.000
   30  interp   29.080  -0.920      0.000
   35  interp   33.480  -1.520      0.000
   40  interp   38.120  -1.880      0.000
   45  interp   42.950  -2.050      0.000
   50 trained   47.830  -2.170      0.000
   55  interp   53.720  -1.280      0.000
   60  interp   59.450  -0.550      0.000
   65  interp   64.940  -0.060      0.000
   70  interp   70.040   0.040      0.000
   75 trained   74.700  -0.300      0.000
   80  interp   80.930   0.930      0.000
   85  interp   85.920   0.920      0.000
   90 trained   89.830  -0.170      0.000
   95 trained   95.140   0.140      0.000

=== precip WET-forecast subset  (hres>=1mm, n=318,037) ===
 rung    kind  cov_pct  err_pp  flat_frac
    5 

In [7]:
# --- regime-conditional worst-rung error: within forecast-magnitude and band-width
#     quintiles (Gate 1: <=4pp in any bucket with n>=5000) -----------------------
def worst_rung_by_bucket(df, key_series, label):
    band = df[BAND_COLS].to_numpy(); truth = df.truth_abs.to_numpy()
    buckets = pd.qcut(key_series, 5, duplicates="drop")
    recs = []
    for bucket, idx in df.groupby(buckets, observed=True).indices.items():
        if len(idx) < 5000:                                   # Gate 1 judges n>=5000 buckets
            continue
        sub_band, sub_truth = band[idx], truth[idx]
        worst, at_rung = 0.0, None
        for r in RUNGS:
            err = abs(float(np.mean(sub_truth <= inv_cdf_col(sub_band, BAND_P, r / 100))) - r / 100) * 100
            if err > worst:
                worst, at_rung = err, r
        recs.append(dict(regime=label, bucket=str(bucket), n=len(idx),
                         worst_err_pp=round(worst, 2), at_rung=at_rung))
    return pd.DataFrame(recs)

regime_tables = {}
for name, df in eval_bands.items():
    tbl = pd.concat([worst_rung_by_bucket(df, df.mid, "magnitude"),       # forecast magnitude
                     worst_rung_by_bucket(df, df.q90 - df.q10, "band_width")],
                    ignore_index=True)
    regime_tables[name] = tbl
    print(f"\n=== {name}: worst rung error within regime buckets ===")
    print(tbl.to_string(index=False))


=== tmax: worst rung error within regime buckets ===
    regime           bucket      n  worst_err_pp  at_rung
 magnitude (-14.671, 22.56] 157096         2.150       20
 magnitude   (22.56, 27.72] 157017         1.880       20
 magnitude   (27.72, 30.33] 157171         2.160       20
 magnitude   (30.33, 33.07] 156408         2.640       20
 magnitude   (33.07, 51.26] 156728         3.790       40
band_width    (0.549, 1.99] 158669         4.390       20
band_width     (1.99, 2.27] 156094         3.460       35
band_width     (2.27, 2.51] 156596         2.990       20
band_width     (2.51, 2.77] 158050         2.370       35
band_width     (2.77, 9.23] 155011         2.570       25



=== tmin: worst rung error within regime buckets ===
    regime           bucket      n  worst_err_pp  at_rung
 magnitude (-21.881, 13.12] 157156         3.170       80
 magnitude   (13.12, 18.78] 156877         2.600       80
 magnitude   (18.78, 22.84] 156938         3.290       80
 magnitude   (22.84, 25.12] 156849         5.150       65
 magnitude   (25.12, 37.44] 156600         5.320       65
band_width    (0.619, 1.62] 159255         7.900       65
band_width     (1.62, 2.06] 155080         5.180       80
band_width     (2.06, 2.43] 156325         3.290       80
band_width      (2.43, 2.9] 157606         1.780       80
band_width      (2.9, 8.51] 156154         1.480       35



=== precip: worst rung error within regime buckets ===
    regime         bucket      n  worst_err_pp  at_rung
 magnitude (-0.001, 4.99] 627664        67.120        5
 magnitude (4.99, 333.45] 156756         2.070       80
band_width  (-0.001, 4.5] 470881        83.080        5
band_width   (4.5, 13.68] 156725        18.750        5
band_width (13.68, 85.17] 156814         1.850       80



=== wind: worst rung error within regime buckets ===
    regime        bucket      n  worst_err_pp  at_rung
 magnitude (0.439, 2.54] 157153         2.830       65
 magnitude  (2.54, 3.25] 157069         1.720       80
 magnitude  (3.25, 4.01] 156504         1.710       80
 magnitude  (4.01, 5.04] 157352         2.010       80
 magnitude (5.04, 35.48] 156342         2.450       40
band_width  (0.229, 1.2] 157432         2.790       65
band_width   (1.2, 1.43] 156339         2.260       80
band_width  (1.43, 1.66] 158563         1.380       80
band_width  (1.66, 1.97] 155698         1.470       80
band_width   (1.97, 6.4] 156388         1.340       35



=== dewpt: worst rung error within regime buckets ===
    regime          bucket      n  worst_err_pp  at_rung
 magnitude (-29.751, 9.16] 157118         3.100       50
 magnitude   (9.16, 14.47] 156822         1.310       65
 magnitude  (14.47, 20.04] 156727         1.990       25
 magnitude   (20.04, 23.5] 157273         4.030       50
 magnitude    (23.5, 29.7] 156480         3.260       50
band_width   (0.149, 1.12] 157342         3.180       50
band_width    (1.12, 1.55] 159060         3.510       80
band_width    (1.55, 1.99] 154257         2.300       50
band_width     (1.99, 2.7] 157472         2.040       50
band_width     (2.7, 9.76] 156289         2.920       50


In [8]:
# --- Gate 1 verdict per variable ------------------------------------------------
GATE_MARGINAL_PP = 2.0     # tmax/tmin/wind: every rung within 2pp marginal
GATE_REGIME_PP   = 4.0     # ... and within 4pp in every n>=5000 regime bucket
GATE_PRECIP_PP   = 4.0     # precip: wet-forecast subset within 4pp

recs = []
for name in ("tmax", "tmin", "wind", "dewpt"):
    marg = rung_tables[name].err_pp.abs().max()
    reg  = regime_tables[name].worst_err_pp.max() if len(regime_tables[name]) else np.nan
    ok = (marg <= GATE_MARGINAL_PP) and (np.isnan(reg) or reg <= GATE_REGIME_PP)
    recs.append(dict(var=name, max_marginal_pp=marg, max_regime_pp=reg,
                     verdict="PASS" if ok else "FAIL"))

# precip: judged on the WET subset; all-rows point-mass rungs are listed, not failed
wet_marg = rung_tables["precip_wet"].err_pp.abs().max()
flat_rungs = rung_tables["precip"].query("flat_frac > 0.5").rung.tolist()
recs.append(dict(var="precip", max_marginal_pp=wet_marg,
                 max_regime_pp=(regime_tables["precip"].worst_err_pp.max()
                                if len(regime_tables["precip"]) else np.nan),
                 verdict="PASS" if wet_marg <= GATE_PRECIP_PP else "FAIL"))

verdict = pd.DataFrame(recs).set_index("var")
print("Gate 1 verdict — max |coverage - rung|, percentage points")
print("(precip max_marginal_pp is the WET subset; regime col shown for context)\n")
print(verdict.to_string())
print(f"\nprecip all-rows rungs on a point-mass segment (flat_frac>0.5), listed not failed: {flat_rungs}")

Gate 1 verdict — max |coverage - rung|, percentage points
(precip max_marginal_pp is the WET subset; regime col shown for context)

        max_marginal_pp  max_regime_pp verdict
var                                           
tmax              2.400          4.390    FAIL
tmin              3.750          7.900    FAIL
wind              1.780          2.830    PASS
dewpt             2.170          4.030    FAIL
precip           10.180         83.080    FAIL

precip all-rows rungs on a point-mass segment (flat_frac>0.5), listed not failed: [5, 10, 15, 20, 25, 30, 35, 40, 45, 50, 55, 60, 65, 70]


## Reading it — Gate 1 (for the reviewing session)

**What each table is.** `err_pp` = empirical coverage minus the rung, in
percentage points (`+` = the CDF over-covers: the settled value lands *below* the
rung more often than the rung claims). `kind=trained` rungs (5/10/25/50/75/90/95) are
Level-0 sanity anchors — they should roughly reproduce
`cqr_per_level_validity.ipynb` (here they are re-measured out-of-sample on the
eval half, so expect a little more noise). As of the q9 retrain that anchor set
now includes **25 and 75**, which used to be probit shoulders. The
`kind=interp` rungs (15,20,30,…,85) are the news: all are linear interpolations
*between* the 9 trained points. `flat_frac` = fraction of rows whose
bracketing CDF segment is degenerate (a point mass) — precip only.

**Gate 1 — PASS if:**

- **tmax / tmin / wind / dewpt** — every rung `|err_pp| ≤ 2` marginal, **and**
  every regime bucket (magnitude- or width-quintile, `n ≥ 5000`) worst rung
  `≤ 4 pp`. Dew point is a continuous °C variable and is judged on exactly the
  same thresholds as the two temperatures — it is new in `debias-v10`, so read
  its row against tmax's and tmin's, which are what the card serves today.
- **precip** — the **wet-forecast subset** (`hres ≥ 1mm`) within **4 pp**;
  all-rows deviations that sit on a point-mass segment (the listed
  `flat_frac > 0.5` rungs) are the known zero-inflation artifact — they are
  **listed, not failed**.

**FAIL → stop.** Notebook 2 is pointless until this passes. Report which
rungs/regimes miss and by how much, and recommend one of: add trained heads at the
failing rungs, recalibrate the CQR shifts, or coarsen the card's 5% rounding.

**Comparison to the 7-level run.** The rungs between 10 and 90 should improve or
hold: two of the points bounding them are now fitted rather than assumed
normal-ish. A *degradation* there points at the retrain, not the interpolation.

**Scope caveat.** Bands here are the raw ML-table path for *all* cells, including
the small gated set (`gated` column: tmax/tmin/wind ≪ 1%, precip a few % of eval
rows) whose production band uses a different, already-validated empirical
mechanism. Gating does not move the temp/wind read; for precip it is folded into
the dry-day framing above. **Notebook 2 excludes gated cells** from the
end-to-end claim check.